# CS781 Phase 1 — Resumable decoder-only Transformer training

This version preserves the project's existing Phase-4 data foundation and streaming JSONL pipeline:
- `legal_bpe_32k.json` tokenizer
- train/validation manifests
- byte-offset `source_index.jsonl`
- `WorkerSafeDocumentAwarePacker` (does not concatenate the whole corpus into RAM)
- randomly initialized GPT-2-style decoder model using the original `MODEL_CONFIG`

**Target:** train to optimizer step **3,000** (set `TOTAL_STEPS = 2_000` if you want the shorter target). Local checkpoints are written frequently; the latest checkpoint is uploaded to a private Kaggle Dataset periodically so a new Kaggle session can resume.

**Before running:** attach the existing Phase-4 artifact dataset as notebook input. Create a private Kaggle Dataset named `cs781-phase1-checkpoints` once, with a `dataset-metadata.json` matching the slug below, and add Kaggle Secrets `KAGGLE_USERNAME` and `KAGGLE_KEY`. If the checkpoint dataset already exists, keep it attached as an input too.

Run the cells top-to-bottom. Do not run the old notebook's one-off training-step or hard-coded “resume from step 700” cells.


In [ ]:
# Install only if the environment is missing these packages.
%pip -q install transformers tokenizers


In [ ]:
# ============================================================
# CELL 1 — ENVIRONMENT
# ============================================================

import os
import gc
import json
import math
import time
import random
from pathlib import Path

import torch
from torch.utils.data import IterableDataset, DataLoader, get_worker_info

print("=" * 80)
print("CS781 PHASE 1 — TRAINING ENVIRONMENT")
print("=" * 80)

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print(
        "VRAM    :",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device  :", DEVICE)

assert torch.cuda.is_available(), "CUDA GPU is required for this run."

print("\n✓ Environment ready")

In [ ]:
# ============================================================
# CELL 2 — PHASE-1 ARTIFACT PATHS
# ============================================================

CANDIDATES = [
    Path("/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"),
    Path("/kaggle/input/datasets/oipranesh/cs781-phase4-final-recovery"),
]

CHECKPOINT = None

for candidate in CANDIDATES:
    if (
        (candidate / "manifests/train_manifest.jsonl").exists()
        and
        (candidate / "tokenizer/legal_bpe_32k.json").exists()
    ):
        CHECKPOINT = candidate
        break

assert CHECKPOINT is not None, "Phase-1 training artifacts were not found."

TRAIN_MANIFEST = CHECKPOINT / "manifests/train_manifest.jsonl"
VAL_MANIFEST = CHECKPOINT / "manifests/validation_manifest.jsonl"
TEST_MANIFEST = CHECKPOINT / "manifests/test_manifest.jsonl"
TOKENIZER_PATH = CHECKPOINT / "tokenizer/legal_bpe_32k.json"

print("=" * 80)
print("PHASE-1 ARTIFACTS")
print("=" * 80)

print("Checkpoint :", CHECKPOINT)
print("Train      :", TRAIN_MANIFEST)
print("Validation :", VAL_MANIFEST)
print("Test       :", TEST_MANIFEST)
print("Tokenizer  :", TOKENIZER_PATH)

assert TRAIN_MANIFEST.exists()
assert VAL_MANIFEST.exists()
assert TEST_MANIFEST.exists()
assert TOKENIZER_PATH.exists()

print("\n✓ All required artifacts found")

In [ ]:
# ============================================================
# CELL 3 — TOKENIZER
# ============================================================

from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

VOCAB_SIZE = tokenizer.get_vocab_size()

PAD_ID = tokenizer.token_to_id("[PAD]")
UNK_ID = tokenizer.token_to_id("[UNK]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")

print("=" * 80)
print("TOKENIZER")
print("=" * 80)

print("Vocabulary :", VOCAB_SIZE)
print("PAD        :", PAD_ID)
print("UNK        :", UNK_ID)
print("BOS        :", BOS_ID)
print("EOS        :", EOS_ID)

assert VOCAB_SIZE == 32000
assert PAD_ID is not None
assert BOS_ID is not None
assert EOS_ID is not None

print("\n✓ Tokenizer loaded")
print("✓ Vocabulary = 32,000")

In [ ]:
# ============================================================
# CELL 4 — STREAMING SOURCE INDEX
# ============================================================

INDEX_CANDIDATES = [
    Path("/kaggle/working/phase4_streaming_index/train_source_index.jsonl"),
    Path(
        "/kaggle/input/datasets/oipranesh/"
        "cs781-phase4-final-recovery/indexes/source_index.jsonl"
    ),
    CHECKPOINT / "indexes/source_index.jsonl",
]

INDEX_PATH = None

for candidate in INDEX_CANDIDATES:
    if candidate.exists():
        INDEX_PATH = candidate
        break

if INDEX_PATH is not None:

    print("Existing source index found:")
    print(INDEX_PATH)

else:

    print("No existing source index found.")
    print("Building index from the Phase-1 deduplicated corpus...")

    DEDUP_ROOT = CHECKPOINT / "dedup"

    assert DEDUP_ROOT.exists(), DEDUP_ROOT

    INDEX_DIR = Path("/kaggle/working/phase4_streaming_index")
    INDEX_DIR.mkdir(parents=True, exist_ok=True)

    INDEX_PATH = INDEX_DIR / "train_source_index.jsonl"

    # Discover source files referenced by the training manifest.
    source_names = set()

    with open(TRAIN_MANIFEST, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                source_names.add(rec["source_file"])

    print("Manifest source files:", len(source_names))

    def resolve_source(source_file):
        candidates = [
            DEDUP_ROOT / source_file,
            DEDUP_ROOT / Path(source_file).with_suffix(""),
        ]

        for p in candidates:
            if p.exists() and p.is_file():
                return p

        matches = [
            p for p in DEDUP_ROOT.rglob(Path(source_file).name)
            if p.is_file()
        ]

        if len(matches) == 1:
            return matches[0]

        raise FileNotFoundError(
            f"Could not resolve source file: {source_file}"
        )

    source_paths = {
        name: resolve_source(name)
        for name in source_names
    }

    print("\nBuilding byte-offset index...")

    total_records = 0

    with open(INDEX_PATH, "w", encoding="utf-8") as out:

        for source_name, source_path in source_paths.items():

            print("Indexing:", source_name)

            with open(
                source_path,
                "rb",
                buffering=1024 * 1024,
            ) as src:

                while True:

                    offset = src.tell()
                    raw = src.readline()

                    if not raw:
                        break

                    try:
                        obj = json.loads(raw)
                    except Exception as e:
                        raise RuntimeError(
                            f"Invalid JSON at {source_path}:{offset}"
                        ) from e

                    text = obj.get("text")

                    if not isinstance(text, str):
                        continue

                    text_hash = __import__("hashlib").sha256(
                        text.encode("utf-8")
                    ).hexdigest()

                    out.write(
                        json.dumps(
                            {
                                "text_hash": text_hash,
                                "source_file": source_name,
                                "byte_offset": offset,
                                "byte_length": len(raw),
                            },
                            separators=(",", ":"),
                        )
                        + "\n"
                    )

                    total_records += 1

            print("  indexed")

    print("\nTotal indexed records:", f"{total_records:,}")

print("\nIndex:", INDEX_PATH)
assert INDEX_PATH.exists()

print("✓ Source index ready")

In [ ]:
# ============================================================
# CELL 5 — WORKER-SAFE DOCUMENT-AWARE STREAMING DATASET
# ============================================================

class WorkerSafeDocumentAwarePacker(IterableDataset):

    def __init__(
        self,
        manifest_path,
        index_path,
        tokenizer,
        context_length=4096,
        checkpoint=None,
        split="train",
    ):
        super().__init__()

        self.manifest_path = str(manifest_path)
        self.index_path = str(index_path)
        self.tokenizer = tokenizer
        self.context_length = int(context_length)
        self.checkpoint = str(checkpoint or CHECKPOINT)
        self.split = split

        # ----------------------------------------------------
        # Load byte-offset index
        # ----------------------------------------------------

        self.index = {}

        with open(self.index_path, "r", encoding="utf-8") as f:

            for line in f:

                if not line.strip():
                    continue

                rec = json.loads(line)

                self.index[rec["text_hash"]] = {
                    "source_file": rec["source_file"],
                    "byte_offset": rec["byte_offset"],
                    "byte_length": rec["byte_length"],
                }

        # ----------------------------------------------------
        # Load manifest
        # ----------------------------------------------------

        self.manifest = []

        with open(self.manifest_path, "r", encoding="utf-8") as f:

            for line in f:

                if line.strip():
                    self.manifest.append(json.loads(line))

        # ----------------------------------------------------
        # Special tokens
        # ----------------------------------------------------

        self.bos_id = int(tokenizer.token_to_id("[BOS]"))
        self.eos_id = int(tokenizer.token_to_id("[EOS]"))
        self.pad_id = int(tokenizer.token_to_id("[PAD]"))

        print(
            f"{split}: "
            f"{len(self.manifest):,} manifest records | "
            f"{len(self.index):,} index records"
        )

    def _read_source_record(self, rec):

        text_hash = rec["text_hash"]

        if text_hash not in self.index:
            raise KeyError(
                f"text_hash not found: {text_hash}"
            )

        idx = self.index[text_hash]

        source_name = idx["source_file"]

        if source_name.endswith(".gz"):
            source_name = source_name[:-3]

        source_path = (
            Path(self.checkpoint)
            / "dedup"
            / source_name
        )

        if not source_path.exists():

            # Robust fallback for unusual artifact layout.
            matches = list(
                Path(self.checkpoint)
                .rglob(Path(source_name).name)
            )

            if len(matches) != 1:
                raise FileNotFoundError(
                    f"Cannot resolve source:\n{source_name}"
                )

            source_path = matches[0]

        with open(source_path, "rb") as f:

            f.seek(idx["byte_offset"])
            raw = f.read(idx["byte_length"])

        raw = raw.rstrip(b"\r\n")

        return json.loads(raw.decode("utf-8"))

    def _extract_text(self, record):

        if isinstance(record, str):
            return record

        for key in ("text", "content", "document", "body"):

            value = record.get(key)

            if isinstance(value, str):
                return value

        raise KeyError(
            f"No text field found. Keys: {list(record.keys())}"
        )

    def _tokenize_document(self, text):

        ids = self.tokenizer.encode(text).ids

        return (
            [self.bos_id]
            + list(map(int, ids))
            + [self.eos_id]
        )

    def _pack_document(self, token_ids):

        for start in range(
            0,
            len(token_ids),
            self.context_length,
        ):

            chunk = token_ids[
                start:start + self.context_length
            ]

            if len(chunk) < self.context_length:

                chunk = chunk + [
                    self.pad_id
                ] * (
                    self.context_length - len(chunk)
                )

            yield chunk

    def __iter__(self):

        worker_info = get_worker_info()

        if worker_info is None:

            worker_id = 0
            num_workers = 1

        else:

            worker_id = worker_info.id
            num_workers = worker_info.num_workers

        # Strided worker partitioning.
        for manifest_idx in range(
            worker_id,
            len(self.manifest),
            num_workers,
        ):

            rec = self.manifest[manifest_idx]

            source_record = self._read_source_record(rec)

            text = self._extract_text(source_record)

            token_ids = self._tokenize_document(text)

            for packed in self._pack_document(token_ids):

                input_ids = torch.tensor(
                    packed,
                    dtype=torch.long,
                )

                labels = input_ids.clone()

                labels[
                    labels == self.pad_id
                ] = -100

                attention_mask = (
                    input_ids != self.pad_id
                ).long()

                yield {
                    "input_ids": input_ids,
                    "labels": labels,
                    "attention_mask": attention_mask,
                }


print("✓ Worker-safe streaming dataset class ready")

In [ ]:
# ============================================================
# CELL 6 — TRAIN / VALIDATION DATASETS + DATALOADERS
# ============================================================

CONTEXT_LENGTH = 4096

BATCH_SIZE = 1
NUM_WORKERS = 0

train_dataset = WorkerSafeDocumentAwarePacker(
    manifest_path=TRAIN_MANIFEST,
    index_path=INDEX_PATH,
    tokenizer=tokenizer,
    context_length=CONTEXT_LENGTH,
    checkpoint=CHECKPOINT,
    split="train",
)

validation_dataset = WorkerSafeDocumentAwarePacker(
    manifest_path=VAL_MANIFEST,
    index_path=INDEX_PATH,
    tokenizer=tokenizer,
    context_length=CONTEXT_LENGTH,
    checkpoint=CHECKPOINT,
    split="validation",
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=False,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=False,
)

print("=" * 80)
print("DATALOADERS READY")
print("=" * 80)

print("Train records :", f"{len(train_dataset.manifest):,}")
print("Val records   :", f"{len(validation_dataset.manifest):,}")
print("Batch size    :", BATCH_SIZE)
print("Workers       :", NUM_WORKERS)
print("Context       :", CONTEXT_LENGTH)

In [ ]:
# ============================================================
# MODEL CONFIGURATION
# ============================================================

from dataclasses import dataclass

# Use the tokenizer that was already loaded, if available.
# Otherwise load it directly from the Phase-4 artifact.
if "tokenizer" not in globals():
    from tokenizers import Tokenizer

    TOKENIZER_PATH = (
        "/kaggle/input/datasets/oipranesh/"
        "cs781-phase4-training-prep/tokenizer/legal_bpe_32k.json"
    )

    tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

# Resolve special-token IDs directly from the tokenizer.
VOCAB_SIZE = tokenizer.get_vocab_size()

PAD_ID = tokenizer.token_to_id("[PAD]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")

print("Vocabulary:", VOCAB_SIZE)
print("PAD_ID:", PAD_ID)
print("BOS_ID:", BOS_ID)
print("EOS_ID:", EOS_ID)

assert VOCAB_SIZE == 32000
assert PAD_ID is not None
assert BOS_ID is not None
assert EOS_ID is not None


@dataclass
class LegalLMConfig:
    vocab_size: int = VOCAB_SIZE
    max_position_embeddings: int = 4096

    hidden_size: int = 768
    intermediate_size: int = 3072

    num_hidden_layers: int = 12
    num_attention_heads: int = 12

    hidden_dropout: float = 0.0
    attention_dropout: float = 0.0

    bos_token_id: int = BOS_ID
    eos_token_id: int = EOS_ID
    pad_token_id: int = PAD_ID


MODEL_CONFIG = LegalLMConfig()

print("\nModel configuration:")
print(MODEL_CONFIG)

In [ ]:
# ============================================================
# CREATE RANDOMLY INITIALIZED DECODER-ONLY TRANSFORMER
# ============================================================

from transformers import GPT2Config, GPT2LMHeadModel

hf_config = GPT2Config(
    vocab_size=MODEL_CONFIG.vocab_size,
    n_positions=MODEL_CONFIG.max_position_embeddings,
    n_ctx=MODEL_CONFIG.max_position_embeddings,

    n_embd=MODEL_CONFIG.hidden_size,
    n_layer=MODEL_CONFIG.num_hidden_layers,
    n_head=MODEL_CONFIG.num_attention_heads,
    n_inner=MODEL_CONFIG.intermediate_size,

    resid_pdrop=MODEL_CONFIG.hidden_dropout,
    embd_pdrop=MODEL_CONFIG.hidden_dropout,
    attn_pdrop=MODEL_CONFIG.attention_dropout,

    bos_token_id=MODEL_CONFIG.bos_token_id,
    eos_token_id=MODEL_CONFIG.eos_token_id,
    pad_token_id=MODEL_CONFIG.pad_token_id,

    use_cache=False,
)

# Fresh model from configuration.
# NO pretrained checkpoint is loaded.
model = GPT2LMHeadModel(hf_config)

print("✓ Model created from scratch")

In [ ]:
# ============================================================
# PARAMETER CHECK
# ============================================================

param_count = sum(p.numel() for p in model.parameters())

print(f"Parameter count: {param_count:,}")

assert 50_000_000 <= param_count <= 300_000_000

print("✓ Parameter count is within Phase-1 requirement")

In [ ]:
# ============================================================
# CELL 9 — GPU + MEMORY OPTIMIZATION
# ============================================================

model = model.to(DEVICE)

model.gradient_checkpointing_enable()
model.config.use_cache = False

print("Device :", next(model.parameters()).device)
print("Gradient checkpointing : enabled")
print("use_cache              :", model.config.use_cache)

assert next(model.parameters()).is_cuda

print("✓ Model ready for T4 training")

In [ ]:
# ============================================================
# TRAINING CONFIGURATION — resumable run
# ============================================================
import os, json, math, time, shutil, signal, subprocess
from pathlib import Path

# Set to 2_000 for a shorter run; 3_000 is the intended target.
TOTAL_STEPS = 3_000
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
GRAD_ACCUM = 8
MAX_GRAD_NORM = 1.0
LOG_EVERY = 10
LOCAL_CHECKPOINT_EVERY = 25
KAGGLE_UPLOAD_EVERY = 250
NUM_WORKERS = 2
BATCH_SIZE = 1
CONTEXT_LENGTH = 4096

OUTPUT_DIR = Path("/kaggle/working/phase1_decoder_transformer")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUTPUT_DIR / "training_log.jsonl"
STATE_PATH = OUTPUT_DIR / "training_state.json"
LATEST_PATH = CHECKPOINT_DIR / "checkpoint_latest.pt"

# Kaggle Dataset slug must already exist for `kaggle datasets version`.
KAGGLE_DATASET_SLUG = "oipranesh/cs781-phase-1-checkpoints"
UPLOAD_DIR = Path("/kaggle/working/phase1_checkpoint_dataset")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Pull Kaggle API credentials from Kaggle Secrets, if configured.
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = _secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = _secrets.get_secret("KAGGLE_KEY")
    print("Kaggle API credentials loaded from Secrets.")
except Exception as e:
    print("Kaggle Secrets not available in this session:", repr(e))
    print("Training can still run locally; Dataset uploads will be skipped until credentials are configured.")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
USE_AMP = torch.cuda.is_available()
AMP_DTYPE = torch.float16

print("Total target steps:", TOTAL_STEPS)
print("Local checkpoint every:", LOCAL_CHECKPOINT_EVERY)
print("Kaggle upload every:", KAGGLE_UPLOAD_EVERY)
print("Output:", OUTPUT_DIR)


In [ ]:
# ============================================================
# CHECKPOINT DISCOVERY + SAFE SAVE + KAGGLE DATASET VERSIONING
# ============================================================
import re

def find_resume_checkpoint():
    # Prefer rolling checkpoint files; use filename step numbers for snapshots
    # so we do not load multi-GB optimizer states merely to inspect their steps.
    candidates = []
    if LATEST_PATH.exists():
        candidates.append((10**18, LATEST_PATH))
    for p in Path("/kaggle/input").rglob("checkpoint_latest.pt"):
        candidates.append((10**18, p))
    for p in Path("/kaggle/input").rglob("checkpoint_step_*.pt"):
        match = re.search(r"checkpoint_step_(\\d+)\\.pt$", p.name)
        if match:
            candidates.append((int(match.group(1)), p))
    if not candidates:
        return None
    # For rolling checkpoint(s), inspect their saved step once each. Numbered
    # snapshots already carry a step in the filename.
    ranked = []
    for named_step, p in candidates:
        if named_step == 10**18:
            try:
                ck = torch.load(p, map_location="cpu", weights_only=False)
                step = int(ck.get("step", -1))
                if step >= 0 and "model_state_dict" in ck:
                    ranked.append((step, p))
            except Exception as e:
                print("Ignoring unreadable checkpoint:", p, repr(e))
        else:
            ranked.append((named_step, p))
    if not ranked:
        return None
    ranked.sort(key=lambda x: x[0])
    step, path = ranked[-1]
    print(f"Found resume checkpoint at step {step}: {path}")
    return path

def save_checkpoint(step, reason="periodic"):
    temp_path = CHECKPOINT_DIR / "checkpoint_tmp.pt"
    payload = {
        "step": int(step),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "model_config": {
            "vocab_size": MODEL_CONFIG.vocab_size,
            "context_length": MODEL_CONFIG.max_position_embeddings,
            "hidden_size": MODEL_CONFIG.hidden_size,
            "num_layers": MODEL_CONFIG.num_hidden_layers,
            "num_heads": MODEL_CONFIG.num_attention_heads,
        },
        "reason": reason,
        "saved_unix_time": time.time(),
    }
    torch.save(payload, temp_path)
    os.replace(temp_path, LATEST_PATH)
    # Keep a numbered snapshot alongside the rolling checkpoint.
    numbered = CHECKPOINT_DIR / f"checkpoint_step_{step:06d}.pt"
    shutil.copy2(LATEST_PATH, numbered)
    # Retain only the two most recent numbered snapshots to limit disk use.
    snapshots = sorted(CHECKPOINT_DIR.glob("checkpoint_step_*.pt"))
    for old in snapshots[:-2]:
        try: old.unlink()
        except OSError: pass
    with open(STATE_PATH, "w", encoding="utf-8") as f:
        json.dump({"step": int(step), "reason": reason, "saved_unix_time": time.time()}, f, indent=2)
    print(f"Saved checkpoint: step={step}, size={LATEST_PATH.stat().st_size / 1e9:.2f} GB ({reason})")

def upload_checkpoint_to_kaggle(step):
    if not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        print("Skipping Dataset upload: Kaggle API credentials are not configured.")
        return False
    if not (UPLOAD_DIR / "dataset-metadata.json").exists():
        metadata = {
            "title": "CS781 Phase 1 Training Checkpoints",
            "id": KAGGLE_DATASET_SLUG,
            "licenses": [{"name": "CC0-1.0"}],
            "private": True,
        }
        (UPLOAD_DIR / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))
    shutil.copy2(LATEST_PATH, UPLOAD_DIR / "checkpoint_latest.pt")
    if LOG_PATH.exists():
        shutil.copy2(LOG_PATH, UPLOAD_DIR / "training_log.jsonl")
    (UPLOAD_DIR / "training_state.json").write_text(json.dumps({
        "step": int(step), "total_steps": int(TOTAL_STEPS), "updated_unix_time": time.time()
    }, indent=2))
    try:
        result = subprocess.run(
            ["kaggle", "datasets", "version", "-p", str(UPLOAD_DIR),
             "-m", f"CS781 Phase 1 checkpoint step {step}", "--dir-mode", "tar"],
            check=True, text=True, capture_output=True, timeout=1800,
        )
        print(result.stdout[-3000:] if result.stdout else "Kaggle Dataset version uploaded.")
        return True
    except Exception as e:
        print("Kaggle upload failed (local checkpoint is safe):", repr(e))
        if getattr(e, "stdout", None): print(e.stdout[-2000:])
        if getattr(e, "stderr", None): print(e.stderr[-3000:])
        return False

# Resume only if a checkpoint is actually attached/found.
# Model architecture must match the checkpoint's recorded configuration.
RESUME_PATH = find_resume_checkpoint()
START_STEP = 0
if RESUME_PATH is not None:
    checkpoint = torch.load(RESUME_PATH, map_location=DEVICE, weights_only=False)
    saved_cfg = checkpoint.get("model_config", {})
    current_cfg = {
        "vocab_size": MODEL_CONFIG.vocab_size,
        "context_length": MODEL_CONFIG.max_position_embeddings,
        "hidden_size": MODEL_CONFIG.hidden_size,
        "num_layers": MODEL_CONFIG.num_hidden_layers,
        "num_heads": MODEL_CONFIG.num_attention_heads,
    }
    if saved_cfg and saved_cfg != current_cfg:
        raise ValueError(f"Checkpoint architecture mismatch. Saved={saved_cfg}; current={current_cfg}")
    model.load_state_dict(checkpoint["model_state_dict"])
    if "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if "scaler_state_dict" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
    START_STEP = int(checkpoint["step"])
    print("Resumed from optimizer step:", START_STEP)
else:
    print("No checkpoint found: starting from randomly initialized model at step 0.")

assert 0 <= START_STEP <= TOTAL_STEPS


In [ ]:
# ============================================================
# TRAIN TO TOTAL_STEPS
# ============================================================
model.train()
train_iterator = iter(train_loader)
running_loss = 0.0
logged_batches = 0
run_started = time.time()

def _save_on_termination(signum, frame):
    print(f"Received signal {signum}; attempting emergency checkpoint save.")
    try:
        save_checkpoint(current_step, reason=f"signal_{signum}")
    except Exception as e:
        print("Emergency save failed:", repr(e))
    raise SystemExit(128 + signum)

# Best effort only; Kaggle may terminate a session without delivering a signal.
try:
    signal.signal(signal.SIGTERM, _save_on_termination)
    signal.signal(signal.SIGINT, _save_on_termination)
except Exception:
    pass

current_step = START_STEP
for step in range(START_STEP + 1, TOTAL_STEPS + 1):
    current_step = step
    optimizer.zero_grad(set_to_none=True)
    accumulated_loss = 0.0

    for micro_step in range(GRAD_ACCUM):
        try:
            batch = next(train_iterator)
        except StopIteration:
            train_iterator = iter(train_loader)
            batch = next(train_iterator)

        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            scaled_loss = loss / GRAD_ACCUM

        scaler.scale(scaled_loss).backward()
        accumulated_loss += float(loss.detach().float().item())

    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
    scaler.step(optimizer)
    scaler.update()

    mean_loss = accumulated_loss / GRAD_ACCUM
    running_loss += mean_loss
    logged_batches += 1

    if step % LOG_EVERY == 0 or step == START_STEP + 1:
        avg_loss = running_loss / max(logged_batches, 1)
        record = {
            "step": step, "loss": avg_loss,
            "perplexity": math.exp(min(avg_loss, 20)),
            "grad_norm": float(grad_norm),
            "elapsed_seconds": time.time() - run_started,
        }
        with open(LOG_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(record) + "\n")
        print(f"step={step:5d}/{TOTAL_STEPS} | loss={avg_loss:.4f} | ppl={record['perplexity']:.2f} | grad={float(grad_norm):.3f} | elapsed={(time.time()-run_started)/60:.1f} min")
        running_loss = 0.0
        logged_batches = 0

    if step % LOCAL_CHECKPOINT_EVERY == 0 or step == TOTAL_STEPS:
        save_checkpoint(step, reason="final" if step == TOTAL_STEPS else "periodic")

    if step % KAGGLE_UPLOAD_EVERY == 0 or step == TOTAL_STEPS:
        # Save immediately before upload. If upload fails, the local checkpoint remains.
        if step % LOCAL_CHECKPOINT_EVERY != 0 and step != TOTAL_STEPS:
            save_checkpoint(step, reason="upload_boundary")
        upload_checkpoint_to_kaggle(step)

print("\nTraining loop finished.")
print("Final step:", current_step)
print("Latest checkpoint:", LATEST_PATH)


In [ ]:
from pathlib import Path

ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]

def find_files(names):
    found = []
    for root in ROOTS:
        if root.exists():
            for name in names:
                found.extend(root.rglob(name))
    return list(dict.fromkeys(p for p in found if p.is_file()))

checkpoints = find_files(["checkpoint_latest.pt"])
tests = find_files(["test.json", "Phase1- Test.json"])
tokenizer_configs = find_files(["tokenizer.json"])
tokenizer_vocab = find_files(["vocab.json"])

print("CHECKPOINTS:")
for p in checkpoints:
    print(p, f"({p.stat().st_size / 1e9:.2f} GB)")

print("\nTEST FILES:")
for p in tests:
    print(p)

print("\nTOKENIZER FILES:")
for p in tokenizer_configs + tokenizer_vocab:
    print(p)

In [ ]:

import json
import torch
from pathlib import Path

CHECKPOINT_PATH = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase-1-checkpoints/checkpoint_latest.pt"
)
TEST_PATH = Path(
    "/kaggle/input/datasets/shubhamjoshi1806/phase1-test-json/test.json"
)
TOKENIZER_PATH = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep/tokenizer/legal_bpe_32k.json"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Verify files exist
for path in [CHECKPOINT_PATH, TEST_PATH, TOKENIZER_PATH]:
    print(f"{path}: exists={path.is_file()}")
    if not path.is_file():
        raise FileNotFoundError(path)

# Inspect test data
with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("\nTest type:", type(test_data).__name__)
print("Number of examples:", len(test_data))

first_key = next(iter(test_data))
print("First ID:", first_key)
print("First record:", str(test_data[first_key])[:1000])

# Load checkpoint
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint keys:", list(checkpoint.keys()))
print("Saved training step:", checkpoint.get("step"))
print("Model config:", checkpoint.get("model_config"))

# Inspect tokenizer JSON structure
with open(TOKENIZER_PATH, "r", encoding="utf-8") as f:
    tokenizer_json = json.load(f)

print("\nTokenizer JSON top-level keys:", list(tokenizer_json.keys()))
print("Tokenizer model type:",
      tokenizer_json.get("model", {}).get("type"))


In [ ]:
from tokenizers import Tokenizer, decoders

bpe = Tokenizer.from_file(str(TOKENIZER_PATH))
bpe.decoder = decoders.ByteLevel()

PAD_ID = bpe.token_to_id("[PAD]")
BOS_ID = bpe.token_to_id("[BOS]")
EOS_ID = bpe.token_to_id("[EOS]")

assert bpe.get_vocab_size() == 32000
assert PAD_ID is not None
assert BOS_ID is not None
assert EOS_ID is not None

print("Vocabulary:", bpe.get_vocab_size())
print("PAD/BOS/EOS:", PAD_ID, BOS_ID, EOS_ID)

test_text = "The Court shall consider the matter."
print("Round-trip:", bpe.decode(bpe.encode(test_text).ids))

In [ ]:
import torch

CONTEXT_LENGTH = 4096
MAX_GENERATION_TOKENS = 300

def generate_128_words(prompt, temperature=0.8, top_k=50, seed=42):
    encoded = bpe.encode(prompt)
    prompt_ids = encoded.ids

    if not prompt_ids:
        prompt_ids = [BOS_ID]

    # Reserve room for generated tokens.
    prompt_ids = prompt_ids[-(CONTEXT_LENGTH - MAX_GENERATION_TOKENS):]
    input_ids = torch.tensor(
        [prompt_ids], dtype=torch.long, device=device
    )
    generated = input_ids.clone()

    # Use a local generator so each example has reproducible sampling.
    rng = torch.Generator(device=device)
    rng.manual_seed(seed)

    with torch.inference_mode():
        for _ in range(MAX_GENERATION_TOKENS):
            context_ids = generated[:, -CONTEXT_LENGTH:]

            outputs = model(
                input_ids=context_ids,
                attention_mask=torch.ones_like(context_ids),
                use_cache=False,
            )
            logits = outputs.logits[:, -1, :] / temperature

            if top_k is not None:
                k = min(top_k, logits.shape[-1])
                values, indices = torch.topk(logits, k, dim=-1)
                filtered = torch.full_like(logits, float("-inf"))
                filtered.scatter_(1, indices, values)
                logits = filtered

            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(
                probs, num_samples=1, generator=rng
            )
            next_id = next_token.item()

            if next_id == EOS_ID:
                break

            generated = torch.cat([generated, next_token], dim=1)

            new_ids = generated[0, input_ids.shape[1]:].tolist()
            text = bpe.decode(new_ids, skip_special_tokens=True).strip()

            if len(text.split()) >= 128:
                break

    new_ids = generated[0, input_ids.shape[1]:].tolist()
    text = bpe.decode(new_ids, skip_special_tokens=True).strip()
    return " ".join(text.split()[:128])

In [ ]:
first_id = next(iter(test_data))

sample_output = generate_128_words(
    test_data[first_id]["text"],
    seed=42,
)

print("ID:", first_id)
print("Generated words:", len(sample_output.split()))
print(sample_output)

assert "Ġ" not in sample_output
assert "Ċ" not in sample_output

In [ ]:
import json
import time
from pathlib import Path

output_path = Path("/kaggle/working/prediction.json")
predictions = {}
start_time = time.time()
total = len(test_data)

for i, (test_id, item) in enumerate(test_data.items(), start=1):
    predictions[str(test_id)] = {
        "text": generate_128_words(
            item["text"],
            seed=42 + i,
        )
    }

    if i % 10 == 0 or i == total:
        print(f"[{i}/{total}] ID={test_id}")

with output_path.open("w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print("Saved:", output_path)
print("Examples:", len(predictions))
print(f"Elapsed: {(time.time() - start_time) / 60:.1f} minutes")

In [ ]:
with output_path.open("r", encoding="utf-8") as f:
    saved = json.load(f)

assert len(saved) == len(test_data)
assert list(saved.keys()) == [str(k) for k in test_data.keys()]

for test_id, item in saved.items():
    assert set(item) == {"text"}
    assert isinstance(item["text"], str)
    assert len(item["text"].split()) <= 128
    assert "Ġ" not in item["text"]
    assert "Ċ" not in item["text"]

print("Validation passed.")
print("Output file:", output_path)
print("Size:", f"{output_path.stat().st_size / 1024:.1f} KB")

## After a Kaggle session ends

1. Check the notebook output for the latest successful `Kaggle Dataset version uploaded` message and step number.
2. In the next session, attach the private `cs781-phase1-checkpoints` Dataset as an input.
3. Run this notebook from the top. It searches attached inputs for `checkpoint_latest.pt`, validates the architecture, and restores model, optimizer, scaler, and step.
4. If the session stops between uploads, you will resume from the most recent uploaded checkpoint, not necessarily the last local step.

Kaggle can terminate a session abruptly; no notebook can guarantee an upload at the exact moment of termination. The frequent local saves plus periodic Dataset versions reduce lost work.
